# COVID-19 Chest X-Ray Classification Using CNN

Classifies chest X-ray images into **Covid**, **Normal**, and **Viral Pneumonia** using a TensorFlow/Keras CNN.

> **Important:** This project is for educational/research purposes only. It is not a medical diagnostic tool.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array

DATA_DIR = Path('../data/Covid19-dataset')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


## 1. Dataset
The expected structure is `data/Covid19-dataset/train` and `data/Covid19-dataset/test`, each containing the three class folders. The dataset is downloaded separately and is not committed to GitHub.


In [ ]:
for split_dir in [TRAIN_DIR, TEST_DIR]:
    print(f'\n{split_dir}')
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            print(f'{class_dir.name}: {len(list(class_dir.glob("*")))} images')


## 2. Sample images


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, class_name in zip(axes, ['Covid', 'Normal', 'Viral Pneumonia']):
    files = list((TRAIN_DIR / class_name).glob('*'))
    img = plt.imread(files[0])
    ax.imshow(img, cmap='gray')
    ax.set_title(class_name)
    ax.axis('off')
plt.tight_layout()


## 3. Image preprocessing and augmentation


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

training_set = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

test_set = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Class mapping:', training_set.class_indices)


## 4. CNN architecture


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 5. Training


In [ ]:
history = model.fit(
    training_set,
    epochs=3,
    validation_data=test_set
)


## 6. Evaluation


In [ ]:
test_loss, test_accuracy = model.evaluate(test_set)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.legend()
plt.tight_layout()


## 7. Save the trained model


In [ ]:
Path('../models').mkdir(exist_ok=True)
model.save('../models/covid_cnn.keras')
print('Saved to ../models/covid_cnn.keras')


## 8. Single-image prediction


In [ ]:
CLASS_NAMES = {v: k for k, v in training_set.class_indices.items()}
image_path = next((TEST_DIR / 'Viral Pneumonia').glob('*'))

img = load_img(image_path, target_size=IMG_SIZE)
arr = img_to_array(img) / 255.0
arr = np.expand_dims(arr, axis=0)

probabilities = model.predict(arr, verbose=0)[0]
predicted_index = int(np.argmax(probabilities))
print('Predicted class:', CLASS_NAMES[predicted_index])
print('Confidence:', f'{probabilities[predicted_index] * 100:.2f}%')
